In [62]:
import pandas as pd
df = pd.read_csv("../data/credit_risk_dataset.csv")

df.head()


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [63]:
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

In [64]:
x = df.drop("loan_status", axis=1)
y = df["loan_status"]

In [65]:
x = pd.get_dummies(x, drop_first=True)


In [66]:
from sklearn.model_selection import train_test_split

x_train , x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [67]:
x_train = x_train.fillna(x_train.mean())
x_test = x_test.fillna(x_test.mean())

In [68]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Gradient Boosting ": GradientBoostingClassifier(),
    "Ada Boost": AdaBoostClassifier(),
    "Naive Bayes": GaussianNB()
}

In [69]:
from sklearn.metrics import accuracy_score

results = {}

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(x_train, y_train)
    prediction = model.predict(x_test)
    accuracy = accuracy_score(y_test, prediction)
    results[name] = accuracy
    print(f"accuracy: {accuracy}/n")

    

Training Logistic Regression...


c:\Users\patil\Documents\Explainable-AI-Loan-Prediction\env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


accuracy: 0.8462482737455884/n
Training Random Forest...
accuracy: 0.9292619303360442/n
Training Decision Tree...
accuracy: 0.8861439312567132/n
Training KNN...
accuracy: 0.8299831210679761/n
Training SVM...
accuracy: 0.8020561608101887/n
Training Gradient Boosting ...
accuracy: 0.9163725640632193/n
Training Ada Boost...
accuracy: 0.8715666717814946/n
Training Naive Bayes...
accuracy: 0.8086542887831825/n


In [70]:
best_model_name = max(results, key=results.get)

print("Best model: ", best_model_name)
print("Best accuracy: ", results[best_model_name])

Best model:  Random Forest
Best accuracy:  0.9292619303360442


In [71]:
best_model = models[best_model_name]

In [72]:
import joblib

joblib.dump(best_model, "../models/best_model.pkl")

['../models/best_model.pkl']

In [73]:
loaded_model = joblib.load("../models/best_model.pkl")

pred = loaded_model.predict(x_test)
print(pred[:5])

[0 0 0 1 1]


In [74]:
param_grids = {
    "Logistic Regression": {
    "C": [0.01, 0.1, 1, 10]
    },

    "Random Forest": {
        "n_estimators": [50, 100],
        "max_depth": [5, 10]
    },

    "Decision Tree": {
        "max_depth": [5, 10, 20]
    },

    "KNN": {
        "n_neighbors": [3, 5, 7]
    },

    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"]
    },

    "Gradient Boosting": {
        "n_estimators": [50, 100],
        "learning_rate": [0.01, 0.1]
    },

    "AdaBoost": {
        "n_estimators": [50, 100],
        "learning_rate": [0.01, 0.1]
    },

    "Naive Bayes": {

    }

}

In [ ]:
from sklearn.model_selection import GridSearchCV

results = {}

best_model = {}
for name, model in models.items():

    print(f"Tuning {name}...")

    grid = GridSearchCV(
        model,
        param_grids[name],
        cv=3,
        scoring = "accuracy"
    )

    grid.fit(x_train, y_train)

    best_model[name] = grid.best_estimator_

    score = grid.best_score_
    
    results[name] = score

    print("Best param: ", grid.best_params_)
    print("score: ", score)
    print()

Tuning Logistic Regression...


c:\Users\patil\Documents\Explainable-AI-Loan-Prediction\env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\patil\Documents\Explainable-AI-Loan-Prediction\env\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
   

Best param:  {'C': 0.1}
score:  0.8497544505831799

Tuning Random Forest...
Best param:  {'max_depth': 10, 'n_estimators': 100}
score:  0.924109883364027

Tuning Decision Tree...
Best param:  {'max_depth': 10}
score:  0.922306629834254

Tuning KNN...
Best param:  {'n_neighbors': 7}
score:  0.8343692449355432

Tuning SVM...


In [ ]:
best_model_name = max(results, key=results.get)

print("Best model: ", best_model_name)
print("Best score: ", results[best_model_name])

In [ ]:
import joblib

best_model = best_models[best_model_name]

joblib.dump(best_model,"../models/best_model.pkl")

SyntaxError: invalid syntax (4113025813.py, line 5)